# Crash!💥 Boom!💥 Bang! 💥🚗🚙💥🚗
**A SQL based exploratory data analysis of traffic accidents in Okinawa, Japan**
  
---   
# <span style="color:#0279f0;">Part 2: The Big Picture</span>
---

In this notebook, we will take a look at big picture numbers for a general view of the data.  
  
  **Analysis Tasks**  
1. **Volume Metrics** - Total accidents, daily averages, highest and lowest counts
2. **Severity Breakdown** - Distribution across fatal, injury, property damage
3. **Temporal Overview** - Monthly, daily, hourly patterns at a glance

**Tools used in this notebook:**
- Pandas – for loading the CSV, quick checks and dataframes for basic data manipulation.
- DuckDB – for all aggregation queries
- Matplotlib / Seaborn – for visualisations

##### 👩🏻‍💻  *<span style="color:#0279f0">Loading data to begin...</span>*

In [62]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from sql.notebook_setup import (
    load_data,
    create_connection,
    print_summary,
)

df = load_data()
conn = create_connection(df)

print_summary(df)


✅ Loaded 2,749 records.
✅ Columns:
 1. data_category
 2. prefecture_code
 3. police_station_code
 4. report_id
 5. accident_details_code
 6. fatalities
 7. injured_persons
 8. route_code
 9. location_code
10. municipality_code
11. occurence_year
12. occurence_month
13. occurence_date
14. occurence_hour
15. occurence_minute
16. day_night_code
17. sunrise_hour
18. sunrise_minute
19. sunset_hour
20. sunset_minute
21. weather_code
22. terrain_code
23. road_condition_code
24. road_config_code
25. traffic_signal_code
26. stop_reg_signage_code_party_a
27. stop_reg_roadmarking_code_party_a
28. stop_reg_signage_code_party_b
29. stop_reg_roadmarking_code_party_b
30. road_width_code
31. road_alignment_code
32. collision_point_code
33. zone_reg_code
34. median_facility_code
35. pedestrian_vehicle_separation_code
36. accident_type_code
37. age_party_a
38. age_party_b
39. party_type_code_party_a
40. party_type_code_party_b
41. vehicle_use_code_party_a
42. vehicle_use_code_party_b
43. vehicle_shape_c

..........  

### **2.1 Volume Metrics**  
How many accidents happened in 2024 and when were the extremes?

####<font color="#0279f0">  
##### 👩🏻‍💻  *<span style="color:#0279f0">Analyzing Volume Metrics...</span>*

In [63]:
# ========================================
# 2.1 Volumne Metrics
# ========================================

#Total Accidents in 2024 and Daily Average Count
query01="""
SELECT 
    COUNT(*) AS Total_Accidents,
    ROUND(COUNT(*)/366,2) AS Daily_Average 
FROM accidents
"""

results01 = conn.execute(query01).fetchdf()
print("\n\n----------------------------------------")
print("Prefecture Total & Daily Average:")
print("----------------------------------------")
display(results01)

#Aggregate accident count by Main Island and Outer Islands (municipal codes: 375, 381 and 382)
query02="""
SELECT
    CASE
        WHEN municipality_code in (375, 381, 382) THEN 'Outer Islands'
        ELSE
            'Main Island'
    END AS Area,
    
    COUNT(*) AS Total_Accidents,
    ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER(),2) AS Percentage

FROM accidents
GROUP BY area
"""

results02 = conn.execute(query02).fetchdf()
print("\n\n----------------------------------------")
print("Municipalities Breakdown:")
print("----------------------------------------")
display(results02)


# Top 10 Highest and Lowest Counts & when the accidents occurred
# Top 10 Days with most no. of accidents
query03="""
SELECT 
    occurence_month AS Month,
    occurence_date as Date,
    COUNT(*) AS Accident_Count,
FROM accidents
GROUP BY occurence_month, occurence_date
ORDER BY Accident_Count DESC
Limit 10
"""

results03=conn.execute(query03).fetchdf()
print("\n\n----------------------------------------")
print("Top 10 Busiest Days:")
print("----------------------------------------")
display(results03)

# Top 10 Days with least no. of accidents
query04="""
SELECT 
    occurence_month AS Month,
    occurence_date as Date,
    COUNT(*) AS Accident_Count,
FROM accidents
GROUP BY occurence_month, occurence_date
ORDER BY Accident_Count ASC
Limit 10
"""

results04=conn.execute(query04).fetchdf()
print("\n\n----------------------------------------")
print("Top 10 Quietest Days:")
print("----------------------------------------")
display(results04)





----------------------------------------
Prefecture Total & Daily Average:
----------------------------------------


,Total_Accidents,Daily_Average
0,2749,7.51




----------------------------------------
Municipalities Breakdown:
----------------------------------------


,Area,Total_Accidents,Percentage
0,Main Island,2745,99.85
1,Outer Islands,4,0.15




----------------------------------------
Top 10 Busiest Days:
----------------------------------------


,Month,Date,Accident_Count
0,12,11,18
1,9,9,17
2,2,15,16
3,6,10,15
4,1,16,15
5,3,5,15
6,9,13,15
7,3,19,14
8,10,4,14
9,10,1,14




----------------------------------------
Top 10 Quietest Days:
----------------------------------------


,Month,Date,Accident_Count
0,12,30,1
1,12,8,1
2,1,28,1
3,12,31,1
4,1,8,1
5,3,10,2
6,1,31,2
7,12,26,2
8,12,23,2
9,2,25,2


### 📊 <span style="color:orange;">Summary of Volume Metrics Insights </span> 
- A total of 2749 accidents occured in 2024
- Daily average was 7.51 accidents.
- Daily accident counts ranged from 1 - 18  

**10 Busiest days**
- Saw 2-3x the daily average
- Moderate spikes < 3x the daily average
- No extreme outliers
- Could spikes in numbers be due to weekend? weather?


**10 Quietest days**
- At least 1 accident occured every day of 2024
- The 5 quietest days saw a single accident each
- 4 of the top 5 quietest days were in end Dec 2024 and 1st week of Jan 2024.
- Could this suggest low traffic patterns during the year end period?

..........  

### **2.2 Severity Breakdown**  
Distribution across fatal accidents and damage. How serious were these accidents?

####<font color="#0279f0">  
##### 👩🏻‍💻  *<span style="color:#0279f0">Analyzing Severity Breakdown...</span>*

In [64]:
# ========================================
# 2.2 Severity Breakdown
# Step 1: Check codes that are in accident_details_code
# Step 2: Distribution of accident codes
# ========================================

#Step 1: Verify accident_details_code values (1=fatal, 2 =injury)
query_check_codes = """
SELECT
    accident_details_code,
    COUNT(*) as accident_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM accidents), 2) as percentage,
    SUM(fatalities) as fatalities,
    SUM(injured_persons) as injuries

FROM accidents
GROUP BY accident_details_code
ORDER BY accident_details_code
"""

# Check for accidents with no casualties
query_no_casualties = """
SELECT
    COUNT(*) as accidents_with_no_casualties
FROM accidents
WHERE fatalities = 0 AND injured_persons = 0
"""


code_check = conn.execute(query_check_codes).fetchdf()
no_casualties = conn.execute(query_no_casualties).fetchdf()

print("----------------------------------------")
print("Accident Details Code Distribution:")
print("----------------------------------------")
display(code_check)

print("\n\n----------------------------------------")
print("Accidents with no casualties:")
print("----------------------------------------")
display(no_casualties)



----------------------------------------
Accident Details Code Distribution:
----------------------------------------


,accident_details_code,accident_count,percentage,fatalities,injuries
0,1,43,1.56,44.0,10.0
1,2,2706,98.44,0.0,3220.0




----------------------------------------
Accidents with no casualties:
----------------------------------------


,accidents_with_no_casualties
0,0


### 📊  <span style="color:orange;">Results of Severity Breakdown Analysis </span> 

> 📖 **Code Legend**
> 
> Code 1 → One or more fatalities within 24 hours of the accident.  
> Code 2 → Injury only accident (no fatalities).
>
> **Dataset note:** Only accidents involving casualties are included. Property-damage-only accidents are excluded.

**Code 1**:
- 43 accidents resulted in at least 1 fatality
- Out of 43 fatal accidents, there were 44 fatalities plus 10 injuries.
- 1.56% of all accidents recorded were fatal.
- **Fatal accidents are rare**  
  
**Code 2**:
- 3220 people were injured in accidents recorded here.
- 98.44% of all accidents recorded result in injuries requiring treatment
- **Most accidents cause injuries and are not fatal**  
  

#### <span style="color:orange;">**📊 INSIGHTS**</span>
    
**Total casualties in 2024 are as follows**
- 44 deaths + 3,230 injuries = 3,274 people affected
- Fatal accidents had multiple victims; 44 deaths + 10 injuries from just 43 fatal accidents
- Fatal accidents are rare. 1.56% of accidents with casualties were fatal in 2024
- Conversely, 98.44% of accidents with casualties involved non-fatal injuries only.

..........  

### **2.3 Temporal Overview**  
When do accidents happen? Let's have a brief look at monthly, weekly and hourly patterns.

##### 👩🏻‍💻  *<span style="color:#0279f0">Analyzing Temporal Patterns...</span>*

In [65]:
# ========================================
# 2.3 Temporal 
# Accidents by Month, Day of the Week, and Hour
# ========================================

#Getting total number of accident in 2024 to calculate percentages below
query_total = "SELECT COUNT(*) AS total FROM accidents"
total_df = conn.execute(query_total).fetchdf()
total_accidents = total_df.loc[0, "total"]
print(f"Total accidents for percentage calculations: {total_accidents}")

total = total_df.loc[0, "total"]

# Accidents by month
query_monthly = f"""
SELECT
    occurence_month as month,
    COUNT(*) as accident_count,
    ROUND(COUNT(*) * 100.0 / {total_accidents}, 2) as percentage
FROM accidents
GROUP BY occurence_month
ORDER BY accident_count
"""

monthly = conn.execute(query_monthly).fetchdf()
print("----------------------------------------")
print("Monthly Distribution:")
print("----------------------------------------")
display(monthly)


#Accidents by day of week
query_day_of_week = f"""
SELECT
    CASE occurence_day_code
        WHEN 1 THEN 'Sun'
        WHEN 2 THEN 'Mon'
        WHEN 3 THEN 'Tue'
        WHEN 4 THEN 'Wed'
        WHEN 5 THEN 'Thu'
        WHEN 6 THEN 'Fri'
        WHEN 7 THEN 'Sat' 
    END AS day_of_week,
    COUNT(*) as accident_count,
    ROUND(COUNT(*)*100.0 / {total_accidents}, 2) as percentage
FROM accidents
GROUP BY occurence_day_code
ORDER BY accident_count
"""
day_of_week = conn.execute(query_day_of_week).fetchdf()
print("\n\n----------------------------------------")
print("Day of Week Distribution:")
print("----------------------------------------")
display(day_of_week)


#Accidents by Hour
query_hourly = f"""
SELECT
    occurence_hour AS hour,
    COUNT (*) AS accident_count,
    ROUND(COUNT(*) * 100.0 / {total_accidents}, 2) as percentage
FROM accidents
GROUP BY occurence_hour
ORDER BY accident_count
"""
hourly = conn.execute(query_hourly).fetchdf()
print("\n\n----------------------------------------")
print("Hourly Distribution:")
print("----------------------------------------")
display(hourly)



Total accidents for percentage calculations: 2749
----------------------------------------
Monthly Distribution:
----------------------------------------


,month,accident_count,percentage
0,12,174,6.33
1,1,207,7.53
2,11,214,7.78
3,5,220,8.00
4,2,225,8.18
5,4,228,8.29
6,6,237,8.62
7,3,242,8.80
8,10,246,8.95
9,8,249,9.06




----------------------------------------
Day of Week Distribution:
----------------------------------------


,day_of_week,accident_count,percentage
0,Sun,308,11.20
1,Mon,393,14.30
2,Sat,394,14.33
3,Fri,409,14.88
4,Wed,411,14.95
5,Thu,412,14.99
6,Tue,422,15.35




----------------------------------------
Hourly Distribution:
----------------------------------------


,hour,accident_count,percentage
0,4,16,0.58
1,1,21,0.76
2,2,25,0.91
3,3,26,0.95
4,0,34,1.24
5,5,37,1.35
6,6,54,1.96
7,23,60,2.18
8,22,62,2.26
9,21,76,2.76


### <span style="color:orange;">📊  Temporal Pattern Insights</span>  

<span style="color:orange;">Monthly Pattern Insights:</span>

At a glance, monthly numbers are relatively evenly distributed, with no drastic changes spotted for any particular month.

Highest accident months:
- September: 256 (9.31%)
- July: 251 (9.13%)
- August: 249 (9.06%)
- Summer months (July - September) see the most accidents
- Summer is peak season for visiting Okinawa. This increase in tourist numbers might translate to increased road activity.

Lowest accident months:
- December: 174 (6.33%)
- January: 207 (7.53%)
- November: 214 (7.78%)
- No surprises here that December is the quietest month. This aligns with our earlier finding of quietest days in late December/early January.

<span style="color:orange;">Day Of Week Pattern Insights:</span>

- Sunday is the safest day. Significantly fewer accidents (308, only 11.20%)  
- Weekdays have the most accidents - Tuesday through Friday are consistently high (~15% each)  
- Tuesday is the busiest - 422 accidents (15.35%)  
- Weekday vs Weekend pattern is clear - weekdays account for ~75% of accidents despite being 5/7 of the week  

Could it be rush hour commuter traffic on weekdays contributing to higher number of accidents? Let's see what the hourly patterns say:

<span style="color:orange;">Hourly Pattern Insights:</span>

RUSH HOUR HYPOTHESIS CONFIRMED!
Clear patterns:
- Morning rush (7-9am): 389 accidents (14.15%)
- Evening rush (4-6pm): 470 accidents (17.10%) !!Highest!!
- Late night (12-5am): Very low activity
- 4am is safest hour: Only 16 accidents (0.58%)

<span style="color:orange;">More Temporal Insights:</span>
- Evening rush hour (5pm) is the most dangerous time with 249 accidents recorded.
- Late night/early morning are safest - minimal traffic
- Rush hour patterns strongly support weekday commuter hypothesis
- Daytime (9am-7pm) is consistently elevated with steady road activity throughout the day.
  
---

#### <span style="color:orange;">**📊  INSIGHTS SUMMARY**</span>  
**The data shows a clear correlation between rush hour periods and higher accident counts.**

Could the following factors be contributing to this pattern?
- Higher traffic volume during rush hour
- Driver fatigue or stress during the morning and evening commute leading to reduced attention or reckless driving
- Road congestion leading to more frequent lane changes or sudden braking

Further statistical analysis is needed to firmly establish a causal link between rush hour commuting and accident frequency. Understanding the true drivers of this pattern will be key to designing effective interventions.